# Phase 1/2 Colab Driver

Rerunnable driver for the position-bias pilot. Every long-running command uses `--resume`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BASE = Path('/content/drive/MyDrive/position_bias_phase1')
PROJECT = BASE / 'position_bias'
DATA = BASE / 'lost-in-the-middle-main' / 'qa_data'
RESULTS = BASE / 'results'
BASE.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
print('PROJECT:', PROJECT)
print('DATA:', DATA)

In [ ]:
import zipfile
data_dir = BASE / 'lost-in-the-middle-main'
data_zip = BASE / 'lost-in-the-middle-main.zip'
if not data_dir.exists() and data_zip.exists():
    with zipfile.ZipFile(data_zip) as zf:
        zf.extractall(BASE)
print('Data dir exists:', data_dir.exists())

In [ ]:
%cd /content/drive/MyDrive/position_bias_phase1/position_bias
!pip install -r requirements.txt
!python -m compileall .
!python -m pytest tests

In [ ]:
# Phase 1 carried-over GPT-2 random-init Jacobian smoke + full run
!python scripts/run_jacobian.py --model gpt2 --init random --seq-len 1024 --n-seqs 2 --seed 0 --out /content/drive/MyDrive/position_bias_phase1/results/jacobian/gpt2_random_L1024_seed0 --resume --smoke
!python scripts/run_jacobian.py --model gpt2 --init random --seq-len 1024 --n-seqs 50 --seed 0 --out /content/drive/MyDrive/position_bias_phase1/results/jacobian/gpt2_random_L1024_seed0 --resume

In [ ]:
import subprocess

MODELS = ['Qwen/Qwen2.5-1.5B', 'Qwen/Qwen2.5-0.5B']
DOC_POSITIONS = {10: [0, 4, 9], 20: [0, 4, 9, 14, 19]}

def model_safe(model):
    return model.replace('/', '__')

def data_file(doc_count, position):
    return DATA / f'{doc_count}_total_documents' / f'nq-open-{doc_count}_total_documents_gold_at_{position}.jsonl.gz'

def qa_out(model, doc_count, position):
    return RESULTS / 'qa' / model_safe(model) / f'{doc_count}_docs' / f'gold_at_{position}'

def jac_out(model, init, doc_count, position):
    return RESULTS / 'jacobian_qa' / model_safe(model) / init / f'{doc_count}_docs' / f'gold_at_{position}'

def run(cmd):
    print(' '.join(str(x) for x in cmd))
    subprocess.run([str(x) for x in cmd], check=True)

In [ ]:
# Task B smoke: exact-prompt Jacobian, 5 examples per run
for model in MODELS:
    for doc_count, positions in DOC_POSITIONS.items():
        for position in positions:
            run(['python', 'scripts/run_jacobian_qa.py', '--model', model, '--init', 'pretrained', '--data-file', data_file(doc_count, position), '--n-examples', 5, '--max-len', 3072, '--seed', 0, '--out', jac_out(model, 'pretrained', doc_count, position), '--resume', '--smoke'])
run(['python', 'scripts/run_jacobian_qa.py', '--model', 'Qwen/Qwen2.5-0.5B', '--init', 'random', '--data-file', data_file(20, 9), '--n-examples', 5, '--max-len', 3072, '--seed', 0, '--out', jac_out('Qwen/Qwen2.5-0.5B', 'random', 20, 9), '--resume', '--smoke'])

In [ ]:
# Task B full: exact-prompt Jacobian, 100 examples per pretrained run + one random comparison
for model in MODELS:
    for doc_count, positions in DOC_POSITIONS.items():
        for position in positions:
            run(['python', 'scripts/run_jacobian_qa.py', '--model', model, '--init', 'pretrained', '--data-file', data_file(doc_count, position), '--n-examples', 100, '--max-len', 3072, '--seed', 0, '--out', jac_out(model, 'pretrained', doc_count, position), '--resume'])
run(['python', 'scripts/run_jacobian_qa.py', '--model', 'Qwen/Qwen2.5-0.5B', '--init', 'random', '--data-file', data_file(20, 9), '--n-examples', 100, '--max-len', 3072, '--seed', 0, '--out', jac_out('Qwen/Qwen2.5-0.5B', 'random', 20, 9), '--resume'])

In [ ]:
# Task A smoke: QA generation, 5 examples per run
for model in MODELS:
    for doc_count, positions in DOC_POSITIONS.items():
        for position in positions:
            run(['python', 'scripts/run_qa.py', '--model', model, '--data-file', data_file(doc_count, position), '--n-examples', 5, '--max-new-tokens', 64, '--seed', 0, '--out', qa_out(model, doc_count, position), '--resume', '--smoke', '--dtype', 'fp16'])

In [ ]:
# Task A full: QA generation, 300 examples per run
for model in MODELS:
    for doc_count, positions in DOC_POSITIONS.items():
        for position in positions:
            run(['python', 'scripts/run_qa.py', '--model', model, '--data-file', data_file(doc_count, position), '--n-examples', 300, '--max-new-tokens', 64, '--seed', 0, '--out', qa_out(model, doc_count, position), '--resume', '--dtype', 'fp16'])

In [ ]:
# Task C: correlation reports
for model in MODELS:
    for doc_count in [10, 20]:
        cmd = ['python', 'scripts/run_correlation.py', '--model', model, '--doc-count', doc_count, '--qa-root', RESULTS / 'qa', '--jacobian-root', RESULTS / 'jacobian_qa', '--out', RESULTS / 'correlation' / model_safe(model) / f'{doc_count}_docs']
        if model == 'Qwen/Qwen2.5-0.5B' and doc_count == 20:
            cmd.extend(['--random-jacobian-dir', jac_out(model, 'random', 20, 9)])
        run(cmd)

In [ ]:
# Task D: regenerate compact RESULTS summary section
import json
from pathlib import Path

results_md = PROJECT / 'RESULTS.md'
marker = '\n## Auto-Regenerated Phase 2 Summary\n'
prefix = results_md.read_text(encoding='utf-8').split(marker)[0] if results_md.exists() else '# Position Bias Phase 1 Results\n'
lines = [marker, '\n']
lines.append('### QA Runs\n\n')
for path in sorted((RESULTS / 'qa').glob('*/*/gold_at_*/summary.json')):
    s = json.loads(path.read_text())
    lines.append(f"- {s['model']} {s['doc_count']}-doc gold@{s['gold_position']}: acc={s['accuracy']} CI={s['bootstrap_ci_95']} verdict={s.get('verdict')}\n")
lines.append('\n### Jacobian-QA Runs\n\n')
for path in sorted((RESULTS / 'jacobian_qa').glob('*/*/*/gold_at_*/summary.json')):
    s = json.loads(path.read_text())
    lines.append(f"- {s['model']} {s['init']} {s['doc_count']}-doc gold@{s['gold_position']}: n={s['n']} median_logmean={s['jac_gold_logmean_median']}\n")
lines.append('\n### Correlations\n\n')
for path in sorted((RESULTS / 'correlation').glob('*/*/correlation_report.json')):
    r = json.loads(path.read_text())
    logit = r['logistic_example_level']
    spear = r['spearman_position_level']
    rel = path.relative_to(BASE)
    lines.append(f"- {r['model']} {r['doc_count']}-doc: logistic_coef={logit.get('coef_jac_gold_logmean')} CI={logit.get('coef_jac_gold_logmean_ci95')} AUC={logit.get('auc')}; Spearman rho={spear.get('rho')} n={spear.get('n')}; report=../{rel}\n")
lines.append('\n### H1 Go/No-Go Template\n\n')
lines.append('- Init U-shape reproduces: TBD from Phase 1 GPT-2 random plot.\n')
lines.append('- Accuracy shows U/recency pattern: TBD from QA tables.\n')
lines.append('- Example-level logistic coefficient positive/significant: TBD from correlation reports.\n')
results_md.write_text(prefix + ''.join(lines), encoding='utf-8')
print(results_md)